# Import Dependencies
Import TensorFlow/Keras and any required utilities for model definition and saving.

In [ ]:
from pathlib import Path

import librosa
import matplotlib.pyplot as plt
import numpy as np
from keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from tensorflow import keras

# Define Voice CNN Architecture
Define the voice model architecture with the layers, shapes, and activation functions.

In [ ]:
def build_voice_model() -> keras.Model:
    return keras.Sequential([
        keras.layers.Input(shape=(40, 40, 1)),
        keras.layers.Conv2D(32, (2, 2), activation="relu", padding="same", name="conv2d_7"),
        keras.layers.Conv2D(48, (2, 2), activation="relu", padding="same", name="conv2d_8"),
        keras.layers.MaxPooling2D((2, 2), name="max_pooling2d_5"),
        keras.layers.BatchNormalization(name="batch_normalization_7"),
        keras.layers.Conv2D(128, (2, 2), activation="relu", padding="same", name="conv2d_9"),
        keras.layers.MaxPooling2D((2, 2), name="max_pooling2d_6"),
        keras.layers.BatchNormalization(name="batch_normalization_8"),
        keras.layers.Dropout(0.25, name="dropout_7"),
        keras.layers.Flatten(name="flatten_3"),
        keras.layers.Dense(128, activation="relu", name="dense_7"),
        keras.layers.Dropout(0.25, name="dropout_8"),
        keras.layers.Dense(64, activation="relu", name="dense_8"),
        keras.layers.BatchNormalization(name="batch_normalization_9"),
        keras.layers.Dropout(0.4, name="dropout_9"),
        keras.layers.Dense(10, activation="softmax", name="dense_9"),
    ])

# Build and Summarize Voice Model
Instantiate the model and call summary() to verify layer order and output shapes.

In [ ]:
voice_model = build_voice_model()
voice_model.summary()

In [ ]:
def extract_mfcc(audio_path, sample_rate=8000, n_mfcc=40, max_frames=40):
    audio, _ = librosa.load(audio_path, sr=sample_rate, mono=True)
    mfcc = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=n_mfcc)
    if mfcc.shape[1] > max_frames:
        mfcc = mfcc[:, :max_frames]
    else:
        mfcc = np.pad(mfcc, ((0, 0), (0, max_frames - mfcc.shape[1])), mode="constant")
    return mfcc

# Download FSDD from the official GitHub link if not already present
fsdd_repo_url = "https://github.com/Jakobovski/free-spoken-digit-dataset/archive/refs/heads/master.zip"

# Colab-friendly base path
if Path("/content").exists():
    base_dir = Path("/content") / "datasets" / "voice"
    base_dir.mkdir(parents=True, exist_ok=True)
else:
    cwd = Path.cwd().resolve()
    base_dir = None
    for parent in [cwd, *cwd.parents]:
        candidate = parent / "Backend" / "model_training" / "datasets" / "voice"
        if candidate.exists():
            base_dir = candidate
            break

    if base_dir is None:
        base_dir = cwd / "datasets" / "voice"
        base_dir.mkdir(parents=True, exist_ok=True)

fsdd_root = base_dir / "free-spoken-digit-dataset-master" / "recordings"

if not fsdd_root.exists():
    import io
    import zipfile
    from urllib.request import urlopen

    print("Downloading FSDD dataset...")
    with urlopen(fsdd_repo_url) as response:
        zip_data = response.read()

    with zipfile.ZipFile(io.BytesIO(zip_data)) as zf:
        zf.extractall(base_dir)

    fsdd_root = base_dir / "free-spoken-digit-dataset-master" / "recordings"

if not fsdd_root.exists():
    raise FileNotFoundError(
        "FSDD recordings not found after download. Expected folder: "
        f"{fsdd_root}"
    )

wav_files = sorted(fsdd_root.glob("*.wav"))
print(f"Found {len(wav_files)} audio files")

features = []
labels = []
for wav_path in wav_files:
    # Filename format: {digit}_{speaker}_{index}.wav
    digit = int(wav_path.stem.split("_")[0])
    mfcc = extract_mfcc(wav_path)
    features.append(mfcc)
    labels.append(digit)

X = np.array(features, dtype="float32")[..., np.newaxis]
Y = to_categorical(np.array(labels), num_classes=10)

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=labels
)

print(f"Train samples: {X_train.shape[0]} | Test samples: {X_test.shape[0]}")

voice_model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

history = voice_model.fit(
    X_train,
    Y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.1,
)

test_loss, test_acc = voice_model.evaluate(X_test, Y_test, verbose=0)
print(f"Test accuracy: {test_acc:.4f}")
print(f"Test loss: {test_loss:.4f}")

# Train Voice Model (FSDD)
Dataset source: https://github.com/Jakobovski/free-spoken-digit-dataset
Load Free Spoken Digit Dataset (FSDD), extract MFCC features, train, and evaluate.

In [ ]:
if "history" in globals():
    epochs = range(1, len(history.history["accuracy"]) + 1)

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history.history["accuracy"], label="train")
    plt.plot(epochs, history.history.get("val_accuracy", []), label="val")
    plt.title("Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history.history["loss"], label="train")
    plt.plot(epochs, history.history.get("val_loss", []), label="val")
    plt.title("Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()

    plt.tight_layout()
    plt.show()
else:
    print("No training history found. Run a training cell to generate plots.")

# Training Curves
Plot accuracy and loss over epochs (requires a training history).

In [ ]:
layer_names = [layer.name for layer in voice_model.layers]
layer_params = [layer.count_params() for layer in voice_model.layers]

print(f"Total parameters: {voice_model.count_params():,}")

plt.figure(figsize=(10, 4))
plt.bar(layer_names, layer_params)
plt.title("Voice Model Parameters by Layer")
plt.xlabel("Layer")
plt.ylabel("Params")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# Model Metrics and Chart
Show parameter counts by layer for a quick architecture check.

# Export Voice Model
Save the voice model to a Keras-compatible file for later training or loading.

In [ ]:
backend_dir = Path.cwd().parent if Path.cwd().name == "model_training" else Path.cwd()
output_path = backend_dir / "models" / "voice_model.keras"
output_path.parent.mkdir(parents=True, exist_ok=True)
voice_model.save(output_path.as_posix())
print(f"Saved voice model to: {output_path}")